In [1]:
from abc import ABC, abstractmethod
from copy import deepcopy
from enum import Enum
import numpy as np
import random
from itertools import combinations
from collections import namedtuple, defaultdict
from random import choice
from copy import deepcopy
import time
from collections import Counter
from typing import Callable, List, Set
from tqdm.auto import tqdm
import numpy as np
import math
import itertools
import time
import pickle
import os
import json
from dataclasses import dataclass

import os
import contextlib
# Rules on PDF and https://cdn.1j1ju.com/medias/a8/5e/26-quixo-rulebook.pdf


class Move(Enum):
    '''
    Selects where you want to place the taken piece. The rest of the pieces are shifted
    '''
    TOP = 0
    BOTTOM = 1
    LEFT = 2
    RIGHT = 3


class Player(ABC):
    def __init__(self) -> None:
        '''You can change this for your player if you need to handle state/have memory'''
        pass

    @abstractmethod
    def make_move(self, game: 'Game') -> tuple[tuple[int, int], Move]:
        '''
        The game accepts coordinates of the type (X, Y). X goes from left to right, while Y goes from top to bottom, as in 2D graphics.
        Thus, the coordinates that this method returns shall be in the (X, Y) format.

        game: the Quixo game. You can use it to override the current game with yours, but everything is evaluated by the main game
        return values: this method shall return a tuple of X,Y positions and a move among TOP, BOTTOM, LEFT and RIGHT
        '''
        pass


class Game(object):
    def __init__(self) -> None:
        self._board = np.full((5, 5), -1, dtype=np.int8)
        self.current_player_idx = 1

    def get_board(self) -> np.ndarray:
        '''
        Returns the board
        '''
        return deepcopy(self._board)

    def get_current_player(self) -> int:
        '''
        Returns the current player
        '''
        return deepcopy(self.current_player_idx)

    def print(self):
        '''Prints the board. -1 are neutral pieces, 0 are pieces of player 0, 1 pieces of player 1'''
        print(self._board)

    def check_winner(self) -> int:
        '''Check the winner. Returns the player ID of the winner if any, otherwise returns -1'''
        # for each row
        player = self.get_current_player()
        winner = -1
        for x in range(self._board.shape[0]):
            # if a player has completed an entire row
            if self._board[x, 0] != -1 and all(self._board[x, :] == self._board[x, 0]):
                # return winner is this guy
                winner = self._board[x, 0]
        if winner > -1 and winner != self.get_current_player():
            return winner
        # for each column
        for y in range(self._board.shape[1]):
            # if a player has completed an entire column
            if self._board[0, y] != -1 and all(self._board[:, y] == self._board[0, y]):
                # return the relative id
                winner = self._board[0, y]
        if winner > -1 and winner != self.get_current_player():
            return winner
        # if a player has completed the principal diagonal
        if self._board[0, 0] != -1 and all(
            [self._board[x, x]
                for x in range(self._board.shape[0])] == self._board[0, 0]
        ):
            # return the relative id
            winner = self._board[0, 0]
        if winner > -1 and winner != self.get_current_player():
            return winner
        # if a player has completed the secondary diagonal
        if self._board[0, -1] != -1 and all(
            [self._board[x, -(x + 1)]
             for x in range(self._board.shape[0])] == self._board[0, -1]
        ):
            # return the relative id
            winner = self._board[0, -1]
        return winner

    def play(self, player1: Player, player2: Player) -> int:
        '''Play the game. Returns the winning player'''
        players = [player1, player2]
        winner = -1
        while winner < 0:
            self.current_player_idx += 1
            self.current_player_idx %= len(players)
            ok = False
            while not ok:
                from_pos, slide = players[self.current_player_idx].make_move(
                    self)
                ok = self.__move(from_pos, slide, self.current_player_idx)
            winner = self.check_winner()
        return winner

    def __move(self, from_pos: tuple[int, int], slide: Move, player_id: int) -> bool:
        '''Perform a move'''
        if player_id not in (0, 1):
            return False
        prev_value = deepcopy(self._board[(from_pos[1], from_pos[0])])
        acceptable = self.__take((from_pos[1], from_pos[0]), player_id)
        if acceptable:
            acceptable = self.__slide((from_pos[1], from_pos[0]), slide)
            if not acceptable:  # restore previous
                self._board[(from_pos[1], from_pos[0])] = deepcopy(prev_value)
        return acceptable

    def __take(self, from_pos: tuple[int, int], player_id: int) -> bool:
        """Checks that {from_pos} is in the border and marks the cell with {player_id}"""
        row, col = from_pos
        from_border = row in (0, 4) or col in (0, 4)
        if not from_border:
            return False  # the cell is not in the border
        if self._board[from_pos] != player_id and self._board[from_pos] != -1:
            return False  # the cell belongs to the opponent
        self._board[from_pos] = player_id
        return True

    @staticmethod
    def __acceptable_slides(from_position: tuple[int, int]):
        """When taking a piece from {from_position} returns the possible moves (slides)"""
        acceptable_slides = [Move.BOTTOM, Move.TOP, Move.LEFT, Move.RIGHT]
        axis_0 = from_position[0]    # axis_0 = 0 means uppermost row
        axis_1 = from_position[1]    # axis_1 = 0 means leftmost column

        if axis_0 == 0:  # can't move upwards if in the top row...
            acceptable_slides.remove(Move.TOP)
        elif axis_0 == 4:
            acceptable_slides.remove(Move.BOTTOM)

        if axis_1 == 0:
            acceptable_slides.remove(Move.LEFT)
        elif axis_1 == 4:
            acceptable_slides.remove(Move.RIGHT)
        return acceptable_slides

    def __slide(self, from_pos: tuple[int, int], slide: Move) -> bool:
        '''Slide the other pieces'''
        if slide not in self.__acceptable_slides(from_pos):
            return False  # consider raise ValueError('Invalid argument value')
        axis_0, axis_1 = from_pos
        # np.roll performs a rotation of the element of a 1D ndarray
        if slide == Move.RIGHT:
            self._board[axis_0] = np.roll(self._board[axis_0], -1)
        elif slide == Move.LEFT:
            self._board[axis_0] = np.roll(self._board[axis_0], 1)
        elif slide == Move.BOTTOM:
            self._board[:, axis_1] = np.roll(self._board[:, axis_1], -1)
        elif slide == Move.TOP:
            self._board[:, axis_1] = np.roll(self._board[:, axis_1], 1)
        return True


In [2]:
class RandomPlayer(Player):
    def __init__(self) -> None:
        super().__init__()
        
    def make_move(self, game: 'Game') -> tuple[tuple[int, int], Move]:
        from_pos = (random.randint(0, 4), random.randint(0, 4))
        move = random.choice([Move.TOP, Move.BOTTOM, Move.LEFT, Move.RIGHT])
        return from_pos, move

In [3]:
def acceptable_slides(from_position: tuple[int, int]):
    """When taking a piece from {from_position} returns the possible moves (slides)"""
    acceptable_slides = [Move.BOTTOM, Move.TOP, Move.LEFT, Move.RIGHT]
    axis_0 = from_position[0]    # axis_0 = 0 means uppermost row
    axis_1 = from_position[1]    # axis_1 = 0 means leftmost column

    if axis_0 == 0:  # can't move upwards if in the top row...
        acceptable_slides.remove(Move.TOP)
    elif axis_0 == 4:
        acceptable_slides.remove(Move.BOTTOM)

    if axis_1 == 0:
        acceptable_slides.remove(Move.LEFT)
    elif axis_1 == 4:
        acceptable_slides.remove(Move.RIGHT)
    return acceptable_slides

In [4]:
class ValuePlayer(Player):
    def __init__(self) -> None:
        super().__init__()
        
        self.pos_ranges = [range(0,5),range(0,5)]
        self.all_pos = list(itertools.product(*self.pos_ranges))
        self.available_pos = []
        for pos in self.all_pos:
            row, col = pos
            from_border = row in (0, 4) or col in (0, 4)
            if from_border:
                self.available_pos.append(pos)
                
        self.set_trajectory()
                
    def set_trajectory(self):
        self.trajectory = []
        
    def get_moves(self, game):
        available_actions = []
        for pos in self.available_pos:
            new_pos = deepcopy((pos[1], pos[0]))
            if game._board[new_pos] != 1 and game._board[new_pos] != -1:
                continue
            available_slides = acceptable_slides(new_pos)
            for slide in available_slides:
                available_actions.append((pos, slide))
        return available_actions
        
        
    def make_move(self, game: 'Game') -> tuple[tuple[int, int], Move]:
        moves = self.get_moves(game)
        from_pos, move = random.choice(moves)
        self.trajectory.append(deepcopy(game._board))
        return from_pos, move

In [5]:
g = Game()
rplayer = RandomPlayer()
eplayer = ValuePlayer()
winner = g.play(rplayer, eplayer)
print(f"Winner: Player {winner}")

Winner: Player 1


In [6]:
@dataclass
class AllPlayers:
    random: Player = RandomPlayer()
    value_player: Player = ValuePlayer()
        
def save_dict(file: dict, path: str) -> dict:
    with open(path, 'w') as out:
        json.dump(file, out)
        
def load_dict(path: str) -> dict:
    with open(path, "r") as f:
        data = json.load(f)
    return data

In [7]:
class ValueClass:
    def __init__(
        self,
        player1: Player,
        player2: Player
    ):
        self.player1 = player1
        self.player2 = player2
        self.set_value_dictionary()
        
    def set_value_dictionary(
        self,
    ):
        self.value_dictionary = defaultdict(float)
        self.model_exist = False
        
    def train(
        self,
        epsilon: float = 0.001,
        steps: int = 5_000_000,
        checkpoints: int = 50_000,
        resume: str = None
    ) -> None:
        
        if not self.model_exist:
            if resume:
                self.value_dictionary = defaultdict(float,load_dict(resume))
            for steps in tqdm(range(steps)):
                g = Game()
                self.player2.set_trajectory()
                final_reward = g.play(self.player1, self.player2)
                for state in self.player2.trajectory:
                    
                    hashable_state = "".join(map(str, list(state.reshape(1, -1)[0])))
                    self.value_dictionary[hashable_state] = round(self.value_dictionary[
                        hashable_state
                    ] + epsilon * (final_reward - self.value_dictionary[hashable_state]), 5)
                
                if steps%checkpoints==0 and steps!=0:
                    self.model_exist = True
                    self.save(file_name=f'cpt-{steps}.json', exist_ok=True)
            self.model_exist = True
        else:
            print('model exists, try to empty the value dictionary first')
        
    def save(
        self,
        models_path: str = './models',
        file_name: str = 'model.json',
        exist_ok: bool = False
        
    ) -> None:
        
        if self.model_exist:
            file_path = f'{models_path}/{file_name}'
            os.makedirs(models_path, exist_ok=True)
            if os.path.isfile(file_path):
                if exist_ok:
                    save_dict(self.value_dictionary, file_path)
                    print(f'Value dictionary have been rewritten in {file_name}')
                else:
                    print('Value dictionary exists')

            else:
                save_dict(self.value_dictionary, file_path)
                print(f'Value dictionary have been created in {file_name}')
            
        else:
            print('Model is not trained, train the model first')
            
        
    def load(
        self,
        file_path: str = './models/value_dictionary.json'
    ) -> None:
        
        assert os.path.isfile(file_path)
        self.model_exist = True
        self.value_dictionary = load_dict(file_path)
        print('value dictionary loaded')
            
        
    def top(
        self,
        n: int = 5,
        reverse: bool = True
    ):
        print(sorted(self.value_dictionary.items(), key=lambda e: e[1], reverse=reverse)[0:n])

In [8]:
vd = ValueClass(AllPlayers.random, AllPlayers.value_player)
vd.train(steps=100)
vd.save()

  0%|          | 0/100 [00:00<?, ?it/s]

Value dictionary exists


In [9]:
vd = ValueClass(AllPlayers.random, AllPlayers.value_player)
vd.load('./models/model.json')

value dictionary loaded


In [8]:
from gym import Env
from gym.spaces import Discrete, Box, Dict, Tuple, MultiDiscrete
from collections import deque
import random
from stable_baselines3.common.env_checker import check_env
def get_available_actions():
    pos_ranges = [range(0,5),range(0,5)]
    all_pos = list(itertools.product(*pos_ranges))
    available_actions = []
    for pos in all_pos:
        row, col = pos
        from_border = row in (0, 4) or col in (0, 4)
        if from_border:
            available_moves = acceptable_slides((pos[1], pos[0]))
            for mov in available_moves:
                available_actions.append((pos, mov))
    return available_actions


def map_actions(discrete_value, available_actions):
    assert discrete_value<len(available_actions)
    assert discrete_value>=0
    assert type(discrete_value)==int
    return available_actions[discrete_value]

2024-01-25 15:20:16.852502: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [9]:
class QuixoEnvV1(Env):
    def __init__(self):
        self.actions = get_available_actions()
        self.action_space = Discrete(len(self.actions))
        self.observation_space = Box(low=-30, high=30, shape=(len(self.actions)+5*5, ), dtype=np.int32)
        self.board = -np.ones((5,5), dtype=np.int8)
    
    
    def slide(self, pos, slide):
        axis_0 = pos[1]
        axis_1 = pos[0]
        
        if slide == Move.RIGHT:
            self.board[axis_0] = np.roll(self.board[axis_0], -1)
        elif slide == Move.LEFT:
            self.board[axis_0] = np.roll(self.board[axis_0], 1)
        elif slide == Move.BOTTOM:
            self.board[:, axis_1] = np.roll(self.board[:, axis_1], -1)
        elif slide == Move.TOP:
            self.board[:, axis_1] = np.roll(self.board[:, axis_1], 1)
        
    def get_moves(self, player):
        a_actions = []
        for i in range(len(self.actions)):
            if self.board[(self.actions[i][0][1], self.actions[i][0][0])]!=player:
                a_actions.append(i)
        
        return a_actions
        
        
    def get_action_results(self):
        rewards = []
        count=0
        neg_count=0
        for i in range(len(self.actions)):
            pos = self.actions[i][0]
            mov = self.actions[i][1]
            if self.board[(pos[1], pos[0])]==0:
                rewards.append(-3)
            else:
                axis_0 = pos[1]
                axis_1 = pos[0]
                cp_board = deepcopy(self.board)
                cp_board[(pos[1], pos[0])] = 0
                
                if mov == Move.RIGHT:
                    cp_board[axis_0] = np.roll(cp_board[axis_0], -1)
                elif mov == Move.LEFT:
                    cp_board[axis_0] = np.roll(cp_board[axis_0], 1)
                elif mov == Move.BOTTOM:
                    cp_board[:, axis_1] = np.roll(cp_board[:, axis_1], -1)
                elif mov == Move.TOP:
                    cp_board[:, axis_1] = np.roll(cp_board[:, axis_1], 1)
                
                winner = self.check_winner(cp_board, 1)
                if winner==0:
                    rewards.append(-1)
                    neg_count += 1
                elif winner==1:
                    rewards.append(5)
                    count +=1
                else:
                    rewards.append(1)
                    
        return rewards, count, neg_count
        
    def step(self, action):
        mapped_action = self.actions[action]
        pos = mapped_action[0]
        mov = mapped_action[1]
        self.ep_count+=1
        
        if self.board[(pos[1], pos[0])] == 0:
            self.score = -5
            self.action_results = len(self.actions) * [0]
            self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
            info = {}
            winner=-1
            info['winner'] = winner
            info['detail'] = "selected position in board taken"
            return self.observation, self.score, self.done,info
        
        self.board[(pos[1], pos[0])] = 1
        self.slide(pos, mov)
    
        winner = self.check_winner(self.board, 1)
        if winner==1:
            if self.ep_count<10:
                self.score = 100
            elif self.ep_count<20:
                self.score = 80
            elif self.ep_count<30:
                self.score = 60
            elif self.ep_count<40:
                self.score = 40
            elif self.ep_count<60:
                self.score = 20  
            else:
                self.score = 10
            self.action_results = len(self.actions) * [0]
            self.done = True
            info = {}
            self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
            info['winner'] = winner
            info['detail'] = "player 1 won with his own move"
            return self.observation, self.score, self.done, info
        
        elif winner==0:
            self.score = -10
            self.action_results = len(self.actions) * [0]
            self.done = True
            info = {}
            self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
            info['winner'] = winner
            info['detail'] = "player 0 won with oponent move"
            return self.observation, self.score, self.done, info
            
            
            
        
        op_actions = self.get_moves(1)
        op_action = random.choice(op_actions)
        op_pos, op_mov = self.actions[op_action]
        self.board[(op_pos[1], op_pos[0])] = 0
        self.slide(op_pos, op_mov)
        winner = self.check_winner(self.board, 0)
        if winner==0:
            self.score = -5
            self.action_results = len(self.actions) * [-10]
            self.done = True
            info = {}
            self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
            info['winner'] = winner
            info['detail'] = "player 0 won with his own move"
            return self.observation, self.score, self.done, info
        
        elif winner==1:
            self.score = 0
            self.action_results = len(self.actions) * [-2]
            self.done = True
            info = {}
            self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
            info['winner'] = winner
            info['detail'] = "player 1 won with oponent move"
            return self.observation, self.score, self.done, info
        
        
        self.action_results, count, neg_count = self.get_action_results()
        if self.ep_count>=35:
            self.score = 1 + count + (35 - self.ep_count)
        else:
            self.score = 1 + count
        info = {}
        self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
        winner=-1
        info['winner'] = winner
        info['detail'] = "no one won"
        return self.observation, self.score, self.done, info
        

    def check_winner(self, board, player_id) -> int:
        '''Check the winner. Returns the player ID of the winner if any, otherwise returns -1'''
        player = player_id
        winner = -1
        for x in range(board.shape[0]):
            if board[x, 0] != -1 and all(board[x, :] == board[x, 0]):
                winner = board[x, 0]
        if winner > -1 and winner != player:
            return winner
        for y in range(board.shape[1]):
            if board[0, y] != -1 and all(board[:, y] == board[0, y]):
                winner = board[0, y]
        if winner > -1 and winner != player:
            return winner
        if board[0, 0] != -1 and all(
            [board[x, x]
                for x in range(board.shape[0])] == board[0, 0]
        ):
            winner = board[0, 0]
        if winner > -1 and winner != player:
            return winner
        if board[0, -1] != -1 and all(
            [board[x, -(x + 1)]
             for x in range(board.shape[0])] == board[0, -1]
        ):
            winner = board[0, -1]
        return winner
    
    
    
    def reset(self):
        self.done = False
        self.ep_count = 0
        self.score = 0
        self.board = -np.ones((5,5), dtype=np.int32)
        self.action_results = len(self.actions)*[1]
        self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
        return self.observation
    
    def update_board(self, board):
        self.board = deepcopy(board)
        
    def get_obs(self):
        action_results,_,_ = self.get_action_results()
        observation = np.append(self.board.flatten(), np.array(action_results, dtype=np.int32))
        return observation

In [10]:
class QuixoEnv(Env):
    def __init__(self, player: int):
        self.actions = get_available_actions()
        self.action_space = Discrete(len(self.actions))
        self.observation_space = Box(low=-30, high=30, shape=(len(self.actions)+5*5, ), dtype=np.int32)
        self.board = -np.ones((5,5), dtype=np.int8)
        self.player = player
        self.opposite = 0 if self.player==1 else 1
        if self.player==1:
            result = self.op_move()
            assert result==None
    
    def slide(self, pos, slide):
        axis_0 = pos[1]
        axis_1 = pos[0]
        
        if slide == Move.RIGHT:
            self.board[axis_0] = np.roll(self.board[axis_0], -1)
        elif slide == Move.LEFT:
            self.board[axis_0] = np.roll(self.board[axis_0], 1)
        elif slide == Move.BOTTOM:
            self.board[:, axis_1] = np.roll(self.board[:, axis_1], -1)
        elif slide == Move.TOP:
            self.board[:, axis_1] = np.roll(self.board[:, axis_1], 1)
        
    def get_moves(self, player):
        a_actions = []
        for i in range(len(self.actions)):
            if self.board[(self.actions[i][0][1], self.actions[i][0][0])]!= player:
                a_actions.append(i)
        
        return a_actions
        
        
    def get_action_results(self):
        rewards = []
        count=0
        neg_count=0
        for i in range(len(self.actions)):
            pos = self.actions[i][0]
            mov = self.actions[i][1]
            if self.board[(pos[1], pos[0])]==self.player:
                rewards.append(0)
            else:
                axis_0 = pos[1]
                axis_1 = pos[0]
                cp_board = deepcopy(self.board)
                cp_board[(pos[1], pos[0])] = self.opposite
                
                if mov == Move.RIGHT:
                    cp_board[axis_0] = np.roll(cp_board[axis_0], -1)
                elif mov == Move.LEFT:
                    cp_board[axis_0] = np.roll(cp_board[axis_0], 1)
                elif mov == Move.BOTTOM:
                    cp_board[:, axis_1] = np.roll(cp_board[:, axis_1], -1)
                elif mov == Move.TOP:
                    cp_board[:, axis_1] = np.roll(cp_board[:, axis_1], 1)
                
                winner = self.check_winner(cp_board, self.player)
                if winner==self.opposite:
                    rewards.append(-1)
                    neg_count += 1
                elif winner==self.player:
                    rewards.append(2)
                    count +=1
                else:
                    rewards.append(1)
                    
        return rewards, count, neg_count
        
    def step(self, action):
        self.ep_count+=1
        result = self.player_move(action)
        if result:
            return result
        result = self.op_move()
        if result:
            return result
        
        
        self.action_results, count, neg_count = self.get_action_results()
        
        if self.ep_count>=35:
            self.score = 1 + count + (35 - self.ep_count)
        else:
            self.score = 1 + count
            
        info = {}
        self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
        winner=-1
        info['winner'] = winner
        info['detail'] = "no one won"
        return self.observation, self.score, self.done, info

    def player_move(self, action):
        mapped_action = self.actions[action]
        pos = mapped_action[0]
        mov = mapped_action[1]
        
        if self.board[(pos[1], pos[0])] == self.opposite:
            self.score = -5
            self.action_results = len(self.actions) * [0]
            self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
            info = {}
            winner=-1
            info['winner'] = winner
            info['detail'] = "selected position in board taken"
            return self.observation, self.score, self.done,info
        
        
        self.board[(pos[1], pos[0])] = self.player
        self.slide(pos, mov)
    
        winner = self.check_winner(self.board, self.player)
        if winner==self.player:
#             if self.ep_count<10:
#                 self.score = 100
            if self.ep_count<20:
                self.score = 80
            elif self.ep_count<30:
                self.score = 60
            elif self.ep_count<40:
                self.score = 40
            elif self.ep_count<60:
                self.score = 20  
            else:
                self.score = 10
            self.action_results = len(self.actions) * [20]
            self.done = True
            info = {}
            self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
            info['winner'] = winner
            info['detail'] = f"player {self.player} won with his own move"
            return self.observation, self.score, self.done, info
        
        elif winner==self.opposite:
            self.score = -10
            self.action_results = len(self.actions) * [15]
            self.done = True
            info = {}
            self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
            info['winner'] = winner
            info['detail'] = f"player {self.opposite} won with oponent move"
            return self.observation, self.score, self.done, info
            
            
        return None
            
    def op_move(self):
        op_actions = self.get_moves(self.player)
        op_action = random.choice(op_actions)
        op_pos, op_mov = self.actions[op_action]
        self.board[(op_pos[1], op_pos[0])] = self.opposite
        self.slide(op_pos, op_mov)
        winner = self.check_winner(self.board,self.opposite)
        if winner==self.opposite:
            self.score = -5
            self.action_results = len(self.actions) * [-10]
            self.done = True
            info = {}
            self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
            info['winner'] = winner
            info['detail'] = f"player {self.opposite} won with his own move"
            return self.observation, self.score, self.done, info
        
        elif winner==self.player:
            self.score = 0
            self.action_results = len(self.actions) * [-5]
            self.done = True
            info = {}
            self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
            info['winner'] = winner
            info['detail'] = f"player {self.player} won with oponent move"
            return self.observation, self.score, self.done, info
        
        return None
    

        

    def check_winner(self, board, player_id) -> int:
        '''Check the winner. Returns the player ID of the winner if any, otherwise returns -1'''
        player = player_id
        winner = -1
        for x in range(board.shape[0]):
            if board[x, 0] != -1 and all(board[x, :] == board[x, 0]):
                winner = board[x, 0]
        if winner > -1 and winner != player:
            return winner
        for y in range(board.shape[1]):
            if board[0, y] != -1 and all(board[:, y] == board[0, y]):
                winner = board[0, y]
        if winner > -1 and winner != player:
            return winner
        if board[0, 0] != -1 and all(
            [board[x, x]
                for x in range(board.shape[0])] == board[0, 0]
        ):
            winner = board[0, 0]
        if winner > -1 and winner != player:
            return winner
        if board[0, -1] != -1 and all(
            [board[x, -(x + 1)]
             for x in range(board.shape[0])] == board[0, -1]
        ):
            winner = board[0, -1]
        return winner
    
    
    
    def reset(self):
        self.done = False
        self.ep_count = 0
        self.score = 0
        self.board = -np.ones((5,5), dtype=np.int32)
        self.action_results = len(self.actions)*[1]
        if self.player==1:
            result = self.op_move()
            assert result==None
            self.action_results,_,_ = self.get_action_results()

        self.observation = np.append(self.board.flatten(), np.array(self.action_results, dtype=np.int32))
        return self.observation
    
    def update_board(self, board):
        self.board = deepcopy(board)
        
    def get_obs(self):
        action_results,_,_ = self.get_action_results()
        observation = np.append(self.board.flatten(), np.array(action_results, dtype=np.int32))
        return observation

In [11]:
env = QuixoEnv(player=0)
check_env(env)

In [28]:
t=0
observation = env.reset()
tot_reward = 0
while True:
    t += 1
    action = env.action_space.sample()
    observation, reward, done, info = env.step(action)
    if done:
        print("Episode finished after {} timesteps".format(t+1))
        break

print(f"episode length: {t}")
print(f"total reward: {tot_reward}")
print(f"winner: player {info['winner']}")
print(info)
env.close()

Episode finished after 41 timesteps
episode length: 40
total reward: 0
winner: player 1
{'winner': 1, 'detail': 'player 1 won with his own move'}


In [54]:
from stable_baselines3 import DQN, PPO, SAC, TD3, A2C
from stable_baselines3.common.evaluation import evaluate_policy
from tqdm import tqdm
ts = int(5e5)

In [ ]:
for player in range(2):
    policy_kwargs = dict(net_arch=dict(pi=[1024, 512, 256, 128], vf=[32, 32, 32 , 32]))
    env = QuixoEnv(player=player)
    model = PPO("MlpPolicy", env, policy_kwargs=policy_kwargs, verbose=1)
    model.learn(total_timesteps=ts, progress_bar=True)
    model.save(f"quixo-ppo-{player}")
    del model

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


Output()

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 32.4     |
|    ep_rew_mean     | -52.4    |
| time/              |          |
|    fps             | 348      |
|    iterations      | 1        |
|    time_elapsed    | 5        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 33.2        |
|    ep_rew_mean          | -55.6       |
| time/                   |             |
|    fps                  | 312         |
|    iterations           | 2           |
|    time_elapsed         | 13          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011500057 |
|    clip_fraction        | 0.0873      |
|    clip_range           | 0.2         |
|    entropy_loss         | -3.78       |
|    explained_variance   | 0.00733     |
|    learning_rate        | 0.

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 32.2         |
|    ep_rew_mean          | -41.4        |
| time/                   |              |
|    fps                  | 284          |
|    iterations           | 11           |
|    time_elapsed         | 79           |
|    total_timesteps      | 22528        |
| train/                  |              |
|    approx_kl            | 0.0155819375 |
|    clip_fraction        | 0.175        |
|    clip_range           | 0.2          |
|    entropy_loss         | -3.63        |
|    explained_variance   | 0.0643       |
|    learning_rate        | 0.0003       |
|    loss                 | 1.73e+03     |
|    n_updates            | 100          |
|    policy_gradient_loss | -0.0561      |
|    value_loss           | 1.93e+03     |
------------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_m

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 25.2        |
|    ep_rew_mean          | 10.4        |
| time/                   |             |
|    fps                  | 276         |
|    iterations           | 22          |
|    time_elapsed         | 163         |
|    total_timesteps      | 45056       |
| train/                  |             |
|    approx_kl            | 0.018522734 |
|    clip_fraction        | 0.209       |
|    clip_range           | 0.2         |
|    entropy_loss         | -3.41       |
|    explained_variance   | 0.134       |
|    learning_rate        | 0.0003      |
|    loss                 | 576         |
|    n_updates            | 210         |
|    policy_gradient_loss | -0.0622     |
|    value_loss           | 1.21e+03    |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 24.3  

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 21.2        |
|    ep_rew_mean          | 27.8        |
| time/                   |             |
|    fps                  | 270         |
|    iterations           | 32          |
|    time_elapsed         | 241         |
|    total_timesteps      | 65536       |
| train/                  |             |
|    approx_kl            | 0.027493872 |
|    clip_fraction        | 0.329       |
|    clip_range           | 0.2         |
|    entropy_loss         | -3.11       |
|    explained_variance   | 0.0457      |
|    learning_rate        | 0.0003      |
|    loss                 | 289         |
|    n_updates            | 310         |
|    policy_gradient_loss | -0.0815     |
|    value_loss           | 528         |
-----------------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 20.9    

---------------------------------------
| rollout/                |           |
|    ep_len_mean          | 19        |
|    ep_rew_mean          | 42.2      |
| time/                   |           |
|    fps                  | 266       |
|    iterations           | 42        |
|    time_elapsed         | 322       |
|    total_timesteps      | 86016     |
| train/                  |           |
|    approx_kl            | 0.0326541 |
|    clip_fraction        | 0.371     |
|    clip_range           | 0.2       |
|    entropy_loss         | -2.88     |
|    explained_variance   | -0.0254   |
|    learning_rate        | 0.0003    |
|    loss                 | 327       |
|    n_updates            | 410       |
|    policy_gradient_loss | -0.0888   |
|    value_loss           | 605       |
---------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 20.6        |
|    ep_rew_mean          | 43.7  

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 19.3        |
|    ep_rew_mean          | 52.3        |
| time/                   |             |
|    fps                  | 263         |
|    iterations           | 52          |
|    time_elapsed         | 404         |
|    total_timesteps      | 106496      |
| train/                  |             |
|    approx_kl            | 0.032943387 |
|    clip_fraction        | 0.385       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.66       |
|    explained_variance   | -0.00779    |
|    learning_rate        | 0.0003      |
|    loss                 | 289         |
|    n_updates            | 510         |
|    policy_gradient_loss | -0.0916     |
|    value_loss           | 668         |
-----------------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 16.9    

----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 16.6       |
|    ep_rew_mean          | 53.2       |
| time/                   |            |
|    fps                  | 260        |
|    iterations           | 62         |
|    time_elapsed         | 486        |
|    total_timesteps      | 126976     |
| train/                  |            |
|    approx_kl            | 0.03811813 |
|    clip_fraction        | 0.394      |
|    clip_range           | 0.2        |
|    entropy_loss         | -2.35      |
|    explained_variance   | 0.0689     |
|    learning_rate        | 0.0003     |
|    loss                 | 342        |
|    n_updates            | 610        |
|    policy_gradient_loss | -0.0905    |
|    value_loss           | 668        |
----------------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 16.3       |
|    ep_rew_mean

In [14]:

model = PPO.load(f"quixo-ppo-{player}", env=env)
# mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=10)
winners = 0
trials = 100
done = False
f=0
for i in range(trials):
    obs = env.reset()
    done = False
    while not done:
        action, _states = model.predict(obs, deterministic=True)
        moves = env.get_moves(0)
        if action not in moves:
            action = random.choice(moves)
            f+=1
        obs, rewards, done, info = env.step(action)
    winners += info["winner"]
    
print(f"win percentage: {winners/trials}")
print(f"total wrong actions: {f}")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
win percentage: 0.4
total wrong actions: 911


In [22]:
policy_kwargs = dict(net_arch=[1024, 512, 256, 128])
model = DQN("MlpPolicy", env, policy_kwargs=policy_kwargs, verbose=1,  learning_rate=0.00001)
model.learn(total_timesteps=ts, progress_bar=True)
model.save("quixo-dqn")
del model

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


Output()

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 30.5     |
|    ep_rew_mean      | -50      |
|    exploration_rate | 0.997    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 556      |
|    time_elapsed     | 0        |
|    total_timesteps  | 122      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 27.8     |
|    ep_rew_mean      | -44.8    |
|    exploration_rate | 0.995    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 524      |
|    time_elapsed     | 0        |
|    total_timesteps  | 222      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 27.5     |
|    ep_rew_mean      | -32.8    |
|    exploration_rate | 0.992    |
| time/               |          |
|    episodes       

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 30.9     |
|    ep_rew_mean      | -54.6    |
|    exploration_rate | 0.933    |
| time/               |          |
|    episodes         | 92       |
|    fps              | 599      |
|    time_elapsed     | 4        |
|    total_timesteps  | 2840     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 30.5     |
|    ep_rew_mean      | -52.2    |
|    exploration_rate | 0.931    |
| time/               |          |
|    episodes         | 96       |
|    fps              | 593      |
|    time_elapsed     | 4        |
|    total_timesteps  | 2925     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 30.7     |
|    ep_rew_mean      | -52.5    |
|    exploration_rate | 0.927    |
| time/               |          |
|    episodes       

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 32.7     |
|    ep_rew_mean      | -60.6    |
|    exploration_rate | 0.864    |
| time/               |          |
|    episodes         | 180      |
|    fps              | 604      |
|    time_elapsed     | 9        |
|    total_timesteps  | 5725     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 32.4     |
|    ep_rew_mean      | -60.4    |
|    exploration_rate | 0.861    |
| time/               |          |
|    episodes         | 184      |
|    fps              | 605      |
|    time_elapsed     | 9        |
|    total_timesteps  | 5833     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 32.4     |
|    ep_rew_mean      | -61.2    |
|    exploration_rate | 0.858    |
| time/               |          |
|    episodes       

...

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 58.3     |
|    ep_rew_mean      | -293     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9024     |
|    fps              | 278      |
|    time_elapsed     | 1370     |
|    total_timesteps  | 382004   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 3.79     |
|    n_updates        | 83000    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 57.8     |
|    ep_rew_mean      | -287     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9028     |
|    fps              | 278      |
|    time_elapsed     | 1370     |
|    total_timesteps  | 382148   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 7.16     |
|    n_updates      

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 61.7     |
|    ep_rew_mean      | -415     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9088     |
|    fps              | 278      |
|    time_elapsed     | 1385     |
|    total_timesteps  | 385438   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 3.6      |
|    n_updates        | 83859    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 62.1     |
|    ep_rew_mean      | -430     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9092     |
|    fps              | 278      |
|    time_elapsed     | 1386     |
|    total_timesteps  | 385788   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 5.92     |
|    n_updates      

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 56.2     |
|    ep_rew_mean      | -314     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9152     |
|    fps              | 277      |
|    time_elapsed     | 1400     |
|    total_timesteps  | 389095   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 6.97     |
|    n_updates        | 84773    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 55.9     |
|    ep_rew_mean      | -311     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9156     |
|    fps              | 277      |
|    time_elapsed     | 1401     |
|    total_timesteps  | 389281   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 6.13     |
|    n_updates      

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 57.6     |
|    ep_rew_mean      | -351     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9216     |
|    fps              | 277      |
|    time_elapsed     | 1415     |
|    total_timesteps  | 392961   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 4.79     |
|    n_updates        | 85740    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 56.9     |
|    ep_rew_mean      | -324     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9220     |
|    fps              | 277      |
|    time_elapsed     | 1416     |
|    total_timesteps  | 393167   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 9.57     |
|    n_updates      

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 61       |
|    ep_rew_mean      | -342     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9280     |
|    fps              | 277      |
|    time_elapsed     | 1430     |
|    total_timesteps  | 396950   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 4.74     |
|    n_updates        | 86737    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 59.8     |
|    ep_rew_mean      | -328     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 9284     |
|    fps              | 277      |
|    time_elapsed     | 1431     |
|    total_timesteps  | 397176   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 6.51     |
|    n_updates      

In [23]:
model = DQN.load("quixo-dqn", env=env)
# mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=10)
winners = 0
trials = 100
done = False
f=0
for i in range(trials):
    obs = env.reset()
    done = False
    while not done:
        action, _states = model.predict(obs, deterministic=True)
        moves = env.get_moves(0)
        if action not in moves:
            action = random.choice(moves)
            f+=1
        obs, rewards, done, info = env.step(action)
    winners += info["winner"]
    
print(f"win percentage: {winners/trials}")
print(f"total wrong actions: {f}")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
win percentage: 0.01
total wrong actions: 7


In [24]:
policy_kwargs = dict(net_arch=[1024, 512, 256, 128])

model = A2C("MlpPolicy", env, policy_kwargs=policy_kwargs, verbose=1)
model.learn(total_timesteps=ts, progress_bar=True)
model.save("quixo-atc")
del model

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


Output()

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 28.4     |
|    ep_rew_mean        | -46.5    |
| time/                 |          |
|    fps                | 242      |
|    iterations         | 100      |
|    time_elapsed       | 2        |
|    total_timesteps    | 500      |
| train/                |          |
|    entropy_loss       | -2.64    |
|    explained_variance | 0        |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | -28.4    |
|    value_loss         | 254      |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 34.1     |
|    ep_rew_mean        | -89.2    |
| time/                 |          |
|    fps                | 249      |
|    iterations         | 200      |
|    time_elapsed       | 4        |
|    total_timesteps    | 1000     |
| train/                |          |
|

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 37.9     |
|    ep_rew_mean        | -110     |
| time/                 |          |
|    fps                | 256      |
|    iterations         | 1400     |
|    time_elapsed       | 27       |
|    total_timesteps    | 7000     |
| train/                |          |
|    entropy_loss       | -2.49    |
|    explained_variance | -0.0782  |
|    learning_rate      | 0.0007   |
|    n_updates          | 1399     |
|    policy_loss        | 31.5     |
|    value_loss         | 600      |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 35.7     |
|    ep_rew_mean        | -99.2    |
| time/                 |          |
|    fps                | 254      |
|    iterations         | 1500     |
|    time_elapsed       | 29       |
|    total_timesteps    | 7500     |
| train/                |          |
|

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 62.8     |
|    ep_rew_mean        | -367     |
| time/                 |          |
|    fps                | 282      |
|    iterations         | 2800     |
|    time_elapsed       | 49       |
|    total_timesteps    | 14000    |
| train/                |          |
|    entropy_loss       | -0.995   |
|    explained_variance | 0        |
|    learning_rate      | 0.0007   |
|    n_updates          | 2799     |
|    policy_loss        | -9.29    |
|    value_loss         | 135      |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 70.4     |
|    ep_rew_mean        | -428     |
| time/                 |          |
|    fps                | 285      |
|    iterations         | 2900     |
|    time_elapsed       | 50       |
|    total_timesteps    | 14500    |
| train/                |          |
|

...

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 125      |
|    ep_rew_mean        | -988     |
| time/                 |          |
|    fps                | 391      |
|    iterations         | 73600    |
|    time_elapsed       | 939      |
|    total_timesteps    | 368000   |
| train/                |          |
|    entropy_loss       | -0.0999  |
|    explained_variance | nan      |
|    learning_rate      | 0.0007   |
|    n_updates          | 73599    |
|    policy_loss        | -0       |
|    value_loss         | 0        |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 125      |
|    ep_rew_mean        | -988     |
| time/                 |          |
|    fps                | 391      |
|    iterations         | 73700    |
|    time_elapsed       | 940      |
|    total_timesteps    | 368500   |
| train/                |          |
|

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 125      |
|    ep_rew_mean        | -988     |
| time/                 |          |
|    fps                | 392      |
|    iterations         | 75000    |
|    time_elapsed       | 955      |
|    total_timesteps    | 375000   |
| train/                |          |
|    entropy_loss       | -0.0999  |
|    explained_variance | nan      |
|    learning_rate      | 0.0007   |
|    n_updates          | 74999    |
|    policy_loss        | -0       |
|    value_loss         | 0        |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 125      |
|    ep_rew_mean        | -988     |
| time/                 |          |
|    fps                | 392      |
|    iterations         | 75100    |
|    time_elapsed       | 956      |
|    total_timesteps    | 375500   |
| train/                |          |
|

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 125      |
|    ep_rew_mean        | -988     |
| time/                 |          |
|    fps                | 393      |
|    iterations         | 76400    |
|    time_elapsed       | 970      |
|    total_timesteps    | 382000   |
| train/                |          |
|    entropy_loss       | -0.0999  |
|    explained_variance | nan      |
|    learning_rate      | 0.0007   |
|    n_updates          | 76399    |
|    policy_loss        | -0       |
|    value_loss         | 0        |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 125      |
|    ep_rew_mean        | -988     |
| time/                 |          |
|    fps                | 393      |
|    iterations         | 76500    |
|    time_elapsed       | 971      |
|    total_timesteps    | 382500   |
| train/                |          |
|

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 125      |
|    ep_rew_mean        | -988     |
| time/                 |          |
|    fps                | 394      |
|    iterations         | 77800    |
|    time_elapsed       | 985      |
|    total_timesteps    | 389000   |
| train/                |          |
|    entropy_loss       | -0.0999  |
|    explained_variance | nan      |
|    learning_rate      | 0.0007   |
|    n_updates          | 77799    |
|    policy_loss        | -0       |
|    value_loss         | 0        |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 125      |
|    ep_rew_mean        | -988     |
| time/                 |          |
|    fps                | 394      |
|    iterations         | 77900    |
|    time_elapsed       | 987      |
|    total_timesteps    | 389500   |
| train/                |          |
|

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 125      |
|    ep_rew_mean        | -988     |
| time/                 |          |
|    fps                | 395      |
|    iterations         | 79200    |
|    time_elapsed       | 1002     |
|    total_timesteps    | 396000   |
| train/                |          |
|    entropy_loss       | -0.0999  |
|    explained_variance | nan      |
|    learning_rate      | 0.0007   |
|    n_updates          | 79199    |
|    policy_loss        | -0       |
|    value_loss         | 0        |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 125      |
|    ep_rew_mean        | -988     |
| time/                 |          |
|    fps                | 395      |
|    iterations         | 79300    |
|    time_elapsed       | 1003     |
|    total_timesteps    | 396500   |
| train/                |          |
|

In [ ]:
model = A2C.load("quixo-atc", env=env)
# mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=10)
winners = 0
trials = 100
done = False
f=0
for i in range(trials):
    obs = env.reset()
    done = False
    while not done:
        action, _states = model.predict(obs, deterministic=True)
        moves = env.get_moves(0)
        if action not in moves:
            action = random.choice(moves)
            f+=1
        obs, rewards, done, info = env.step(action)
        
    winners += info["winner"]
    
print(f"win percentage: {winners/trials}")

In [17]:
class DeterministicPlayer(Player):
    def __init__(self) -> None:
        super().__init__()
        self.pos_ranges = [range(0,5),range(0,5)]
        self.all_pos = list(itertools.product(*self.pos_ranges))
        self.available_pos = []
        for pos in self.all_pos:
            row, col = pos
            from_border = row in (0, 4) or col in (0, 4)
            if from_border:
                self.available_pos.append(pos)
        
    def check_winner(self, board, player_id) -> int:
        '''Check the winner. Returns the player ID of the winner if any, otherwise returns -1'''
        player = player_id
        winner = -1
        for x in range(board.shape[0]):
            if board[x, 0] != -1 and all(board[x, :] == board[x, 0]):
                winner = board[x, 0]
        if winner > -1 and winner != player:
            return winner
        for y in range(board.shape[1]):
            if board[0, y] != -1 and all(board[:, y] == board[0, y]):
                winner = board[0, y]
        if winner > -1 and winner != player:
            return winner
        if board[0, 0] != -1 and all(
            [board[x, x]
                for x in range(board.shape[0])] == board[0, 0]
        ):
            winner = board[0, 0]
        if winner > -1 and winner != player:
            return winner
        if board[0, -1] != -1 and all(
            [board[x, -(x + 1)]
             for x in range(board.shape[0])] == board[0, -1]
        ):
            winner = board[0, -1]
        return winner
    
    
    def get_moves(self, game):
        available_actions = []
        for pos in self.available_pos:
            new_pos = deepcopy((pos[1], pos[0]))
            if game._board[new_pos] != game.current_player_idx and game._board[new_pos] != -1:
                continue
            available_slides = acceptable_slides(new_pos)
            for slide in available_slides:
                available_actions.append((pos, slide))
        return available_actions
    
    
    
    def get_action_results(self, game, available_actions):
        rewards = []
        count=0
        neg_count=0
        no_wins = []
        for action in available_actions:
            pos = action[0]
            mov = action[1]
            axis_0 = pos[1]
            axis_1 = pos[0]
            cp_board = deepcopy(game._board)
            cp_board[(pos[1], pos[0])] = game.current_player_idx

            if mov == Move.RIGHT:
                cp_board[axis_0] = np.roll(cp_board[axis_0], -1)
            elif mov == Move.LEFT:
                cp_board[axis_0] = np.roll(cp_board[axis_0], 1)
            elif mov == Move.BOTTOM:
                cp_board[:, axis_1] = np.roll(cp_board[:, axis_1], -1)
            elif mov == Move.TOP:
                cp_board[:, axis_1] = np.roll(cp_board[:, axis_1], 1)

            winner = self.check_winner(cp_board, game.current_player_idx)
            if winner==game.current_player_idx:
                return action
            if winner==-1:
                no_wins.append(action)
                
        
        return no_wins
                    
    
    
    def make_move(self, game: 'Game') -> tuple[tuple[int, int], Move]:
        moves = self.get_moves(game)
        best_actions = self.get_action_results(game, moves)
        if type(best_actions)==list:
            action = random.choice(best_actions)
        else:
            action = best_actions
            
        return action
    


In [52]:
def evaluate(
    eplayer_first: Player,
    eplayer_second: Player,
    opponent: Player = RandomPlayer(),
    enum: int = 10,
):
    win1 = 0
    win2 = 0
    for i in range(enum):
        g = Game()
        winner = g.play(eplayer_first, opponent)
        if winner==0:
            win1+=1

    for i in range(enum):
        g = Game()
        winner = g.play(opponent, eplayer_second)
        if winner==1:
            win2+=1

    print(f"WIN RATIO -- PLAYING SECOND:   {round(win1/enum, 3)}")
    print(f"WIN RATIO -- PLAYING FIRST:    {round(win2/enum, 3)}")
    print(f"TOTAL WIN RATIO:               {round((win2+win1)/(2*enum), 3)}")

In [21]:
eplayer = DeterministicPlayer()
evaluate(eplayer, enum=100)

WIN RATIO -- PLAYING SECOND:   0.84
WIN RATIO -- PLAYING FIRST:    0.85
TOTAL WIN RATIO:               0.845


In [22]:
env = QuixoEnv(player=1)
model = PPO.load("quixo-ppo-1", env=env)
# mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=10)
winners = 0
trials = 100
done = False

f=0
for i in range(trials):
    obs = env.reset()
    done = False
    while not done:
        action, _states = model.predict(obs, deterministic=True)
        moves = env.get_moves(0)
        if action not in moves:
            action = random.choice(moves)
            f+=1
        obs, rewards, done, info = env.step(action)
    winners += info["winner"]
    
print(f"win percentage: {winners/trials}")
print(f"total wrong actions: {f}")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
win percentage: 0.43
total wrong actions: 862


In [48]:
class PPOPlayer(Player):
    def __init__(self, env) -> None:
        super().__init__()
        self.env = env
        self.env.reset()
        self.model = PPO.load(f"quixo-ppo-{self.env.player}", env=self.env)
        self.opposite = 0 if self.env.player==1 else 1
    
    def make_move(self, game: 'Game') -> tuple[tuple[int, int], Move]:
        self.env.update_board(game._board)
        obs = self.env.get_obs()
        action, _ = self.model.predict(obs, deterministic=True)
        self.env.update_board(game._board)
        moves = self.env.get_moves(self.opposite)
        if action not in moves:
            action = random.choice(moves)
        return self.env.actions[action]

In [53]:
opponent = DeterministicPlayer()
with open(os.devnull, "w") as f, contextlib.redirect_stdout(f):
    env0 = QuixoEnv(player=0)
    env1 = QuixoEnv(player=1)
    eplayer_first = PPOPlayer(env0)
    eplayer_second = PPOPlayer(env1)
    
evaluate(eplayer_first, eplayer_second, opponent, enum=30)

WIN RATIO -- PLAYING SECOND:   0.3
WIN RATIO -- PLAYING FIRST:    0.267
TOTAL WIN RATIO:               0.283


In [49]:
import tkinter as tk
from tkinter import ttk, messagebox
from ttkbootstrap import Style


class ButMove:
    def __init__(self):
        self.mov = None
        self.pos = None
        self.pos_selected = False
        self.mov_selected = False
        self.pos_ranges = [range(0,5),range(0,5)]
        self.all_pos = list(itertools.product(*self.pos_ranges))
        self.available_pos = []
        self.current_player = 'X'
        self.game = [['', '', '', '', ''] for _ in range(5)]
        for pos in self.all_pos:
            row, col = pos
            from_border = row in (0, 4) or col in (0, 4)
            if from_border:
                self.available_pos.append(pos)
                
                
    def make_pos(self, i, j):
        if self.pos==(i,j):
            self.game[i][j] = ''
            buttons[i][j].configure(text='')
            self.pos = None
            self.pos_selected = False
            if not self.mov_selected:
                self.deeliminate_mov((i,j))
            
        elif self.pos==None and not self.pos_selected:
            if self.game[i][j] == '' or self.game[i][j] == self.current_player:
                if i in (0,4) or j in (0,4):
                    self.game[i][j] = self.current_player
                    buttons[i][j].configure(text=self.current_player, )
                    self.pos = (i, j)
                    self.pos_selected=True
                    if not self.mov_selected:
                        self.eliminate_mov(self.pos)
                    if self.mov:
                        self.act()
                    
    def deeliminate_mov(self, pos):
        axis_0 = pos[0]
        axis_1 = pos[1]
        
        if axis_0 == 0:  # can't move upwards if in the top row...
            buttons[5][0].configure(text=moves[0], )
        elif axis_0 == 4:
            buttons[6][0].configure(text=moves[1], )

        if axis_1 == 0:
            buttons[7][0].configure(text=moves[2], )
        elif axis_1 == 4:
            buttons[8][0].configure(text=moves[3], ) 
            
            
    def eliminate_mov(self, pos):
        axis_0 = pos[0]
        axis_1 = pos[1]
        
        if axis_0 == 0:  # can't move upwards if in the top row...
            buttons[5][0].configure(text=f"-NOT-SELECTABLE-", )
        elif axis_0 == 4:
            buttons[6][0].configure(text=f"-NOT-SELECTABLE-", )

        if axis_1 == 0:
            buttons[7][0].configure(text=f"-NOT-SELECTABLE-", )
        elif axis_1 == 4:
            buttons[8][0].configure(text=f"-NOT-SELECTABLE-", )
    
    def eliminate_pos(self, mov):
        self.prev_values = deepcopy(game)
        for pos in self.available_pos:
            axis_0 = pos[1]
            axis_1 = pos[0]
            if mov == 0 and axis_0 == 0:
                buttons[axis_0][axis_1].configure(text=f"--", )
            elif mov == 1 and axis_0 == 4:
                buttons[axis_0][axis_1].configure(text=f"--", )

            if mov == 2 and axis_1 == 0:
                buttons[axis_0][axis_1].configure(text=f"--", )
            elif mov == 3 and axis_1 == 4:
                buttons[axis_0][axis_1].configure(text=f"--", )
        
    def deeliminate_pos(self, mov):
        for pos in self.available_pos:
            axis_0 = pos[1]
            axis_1 = pos[0]
            if mov == 0 and axis_0 == 0:
                self.game[axis_0][axis_1] = self.prev_values[axis_0][axis_1]
                buttons[axis_0][axis_1].configure(text=self.prev_values[axis_0][axis_1])
            elif mov == 1 and axis_0 == 4:
                self.game[axis_0][axis_1] = self.prev_values[axis_0][axis_1]
                buttons[axis_0][axis_1].configure(text=self.prev_values[axis_0][axis_1])

            if mov == 2 and axis_1 == 0:
                self.game[axis_0][axis_1] = self.prev_values[axis_0][axis_1]
                buttons[axis_0][axis_1].configure(text=self.prev_values[axis_0][axis_1])
            elif mov == 3 and axis_1 == 4:
                self.game[axis_0][axis_1] = self.prev_values[axis_0][axis_1]
                buttons[axis_0][axis_1].configure(text=self.prev_values[axis_0][axis_1])
                
    def make_mov(self, i,j):
        
        if self.mov==(i,j):
            buttons[5+i][0].configure(text=f"{moves[i]}")
            self.mov = None
            self.mov_selected = False
            if not self.pos_selected:
                self.deeliminate_pos(i)
                
        elif self.mov==None and self.mov!=(i,j) and not self.mov_selected:
            self.mov = (i, j)
            buttons[5+i][0].configure(text=f"{moves[i]}\n-SELECTED-", )
            self.mov_selected = True
            if not self.pos_selected:
                self.eliminate_pos(self.mov[0])
            if self.pos:
                self.act()
            
        
    def act(self):
        sefl.deeliminate_mov(self.pos)
        sefl.deeliminate_pos(self.mov[0])
        row = self.pos[0]
        col = self.pos[1]
        self.game[row][col] = self.current_player
        
        
        buttons[row][col].configure(text=self.current_player)
        self.check_winner()
        self.current_player = "O" if self.current_player == "X" else "X"
        sefl.deeliminate_mov(self.pos)
        sefl.deeliminate_pos(self.mov[0])
        self.mov = None
        self.pos = None
        self.pos_selected = False
        self.mov_selected = False
        
        

    def check_winner(self):
        # Check rows, columns and diagonals
        winning_combinations = (game[0], game[1], game[2],game[3], game[4],
                                [game[i][0] for i in range(5)],
                                [game[i][1] for i in range(5)],
                                [game[i][2] for i in range(5)],
                                [game[i][3] for i in range(5)],
                                [game[i][4] for i in range(5)],
                                [game[i][i] for i in range(5)],
                                [game[i][2 - i] for i in range(5)])

        for combination in winning_combinations:
            if combination[0] == combination[1] == combination[2] == combination[3] == combination[4] != '':
                self.announce_winner(combination[0])

        if all(game[i][j] != '' for i in range(5) for j in range(5)):
            self.announce_winner("Draw")

    def announce_winner(self, player):
        if player == "Draw":
            message = "It's a draw!"
        else:
            message = f"Player {player} wins!"
        messagebox.showinfo("Game Over", message)
        self.reset_game()

    def reset_game(self):
        self.game = [['', '', '', '', ''] for _ in range(5)]
        self.current_player = "X"
        for row in buttons:
            for button in row:
                button.configure(text='')

# Create the main window
root = tk.Tk()
root.title("Tic-Tac-Toe")
style = Style(theme="flatly")

# Create buttons for the game board
buttons = []
mov_obj = ButMove()
for i in range(5):
    row = []
    for j in range(5):
        button = tk.Button(root, text='', width=5,
                            command=lambda i=i, j=j: mov_obj.make_pos(i,j))
        button.grid(row=i, column=j, padx=5, pady=5)
        row.append(button)
    buttons.append(row)

# Initialize the game board and the current player


moves = ['TOP', 'BOTTOM', 'LEFT', 'RIGHT']
actions = {
    moves[0] : Move.TOP,
    moves[1] : Move.BOTTOM,
    moves[2] : Move.LEFT,
    moves[3] : Move.RIGHT,
    
}

for i in range(4):
    row = []
    button = tk.Button(root, text=f'{moves[i]}', width=20,
                        command=lambda i=i, j=7: mov_obj.make_mov(i,j), bg='RED', fg='yellow')
    button.grid(row=i, column=7, padx=40, pady=7)
    row.append(button)
    buttons.append(row)

mov_obj.current_player = "X"
    
root.mainloop()

can't invoke "event" command: application has been destroyed
    while executing
"event generate $w <<ThemeChanged>>"
    (procedure "ttk::ThemeChanged" line 6)
    invoked from within
"ttk::ThemeChanged"


In [35]:
game[i][j]

''